# Convert PDFs to per-page PNGs (PyMuPDF / fitz)

This notebook converts each PDF page to a PNG image and writes the output into a sibling folder named like `filename_pdf/`.

For `docs/filename.pdf`, it creates:
- `docs/filename_pdf/page_001.png`
- `docs/filename_pdf/page_002.png`
- ...


In [22]:
%pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.


In [23]:
from __future__ import annotations

from pathlib import Path
from typing import Optional

import fitz  # PyMuPDF

In [24]:
from pathlib import Path
print(Path("docs").resolve())
print([p.name for p in Path("docs").glob("*.pdf")])

/Users/sach/alrouf-rag-app/data-pipeline/docs
[]


In [25]:
def render_pdf_to_png(
    pdf_path: Path,
    *,
    zoom: float = 2.0,
    dpi: Optional[int] = None,
    overwrite: bool = False,
) -> Path:
    """Render all pages of a PDF to PNG files.

    Returns the output directory path.
    """
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"Not a PDF: {pdf_path}")
    if not pdf_path.exists():
        raise FileNotFoundError(str(pdf_path))

    stem = pdf_path.stem
    out_dir = pdf_path.parent / f"{stem}_pdf"

    sentinel = out_dir / "page_001.png"
    if sentinel.exists() and not overwrite:
        print(f"Skipping (already rendered): {pdf_path.name}")
        return out_dir

    doc = fitz.open(str(pdf_path))
    try:
        out_dir.mkdir(parents=True, exist_ok=True)

        # Prefer zoom. If dpi is provided and zoom <= 0, approximate zoom as dpi/72.
        if dpi is not None and zoom <= 0:
            zoom = dpi / 72.0

        matrix = fitz.Matrix(zoom, zoom)

        print(f"Rendering: {pdf_path.name} -> {out_dir}")
        for page_index in range(doc.page_count):
            page = doc.load_page(page_index)
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            out_path = out_dir / f"page_{page_index + 1:03d}.png"
            pix.save(str(out_path))
    finally:
        doc.close()

    return out_dir

In [26]:
# ----------- Configure your inputs here -----------
# Option A: render all PDFs in a directory
# input_dir = Path("docs")
# pdf_paths = sorted(input_dir.glob("*.pdf"))

# Option B: render a single PDF (robust to notebook working dir)
top_docs_dir = Path("docs")
if not top_docs_dir.exists():
    top_docs_dir = Path("../docs")
if not top_docs_dir.exists():
    raise FileNotFoundError(
        f"Could not find top-level docs directory. Checked: {Path('docs').resolve()} and {Path('../docs').resolve()}"
    )

pdf_path = top_docs_dir / "Alrouf-Medium-Intensity-Aircraft-Warning-Light-AB-ARTX-MI.pdf"
pdf_paths = [pdf_path]

# Rendering options
zoom = 2.0
dpi = None  # set an int if you want to drive scaling via DPI (only used when zoom <= 0)
overwrite = False

# ----------- Run -----------
for p in pdf_paths:
    render_pdf_to_png(p, zoom=zoom, dpi=dpi, overwrite=overwrite)


Rendering: Alrouf-Medium-Intensity-Aircraft-Warning-Light-AB-ARTX-MI.pdf -> ../docs/Alrouf-Medium-Intensity-Aircraft-Warning-Light-AB-ARTX-MI_pdf
